<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Supervised_Learning_lektioner/Lab_3_Lektion_3_Trana_Modell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Supervised Learning – Lektion 3: Träna din första modell!

**Målgrupp:** Gymnasiet, 16 år  
**Tid:** ca 45 minuter  
**Mål:** Träna en Decision Tree-klassificerare, förstå Train/Test split, se modellens prestanda

---

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/  
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

print("✅ Alla bibliotek importerade!")

# Ladda data
iris = load_iris()
X = iris.data
y = iris.target
feature_names = ['sepal_längd', 'sepal_bredd', 'petal_längd', 'petal_bredd']
class_names = ['Setosa', 'Versicolor', 'Virginica']
colors = ['#FF9AA2', '#FFB347', '#87CEEB']

print("✅ Iris-datasetet laddat!")
print(f"   X (features): {X.shape}")
print(f"   y (labels):   {y.shape}")

---
## 🎯 Del 1 – Vad är en modell?

En **modell** i Machine Learning är en **matematisk funktion** som:
1. Tar emot **input** (features)
2. Ger **output** (förutsägelse)

```
INPUT                   MODELL              OUTPUT
[5.1, 3.5, 1.4, 0.2] → [🌳 Besluts- ] → "Setosa"
(sepal L, W, petal L, W)    träd          (förutsagd art)
```

### Beslutsträd (Decision Tree)

Vi använder ett **beslutsträd** – en av de enklaste och mest intuitiva ML-modellerna!

```
           petal_längd < 2.5?
           /              \
         JA               NEJ
          |                |
       SETOSA    petal_bredd < 1.75?
                 /              \
               JA               NEJ
                |                |
           VERSICOLOR         VIRGINICA
```

Trädet ställer JA/NEJ-frågor om features och navigerar ner till ett svar!

---

### 🧩 Quiz 3.1

Följ beslutsträdet ovan för en blomma med: petal_längd=4.5, petal_bredd=1.2

1. Är petal_längd < 2.5? → ___
2. Vilken väg tar vi? →
3. Är petal_bredd < 1.75? → ___
4. Vilken art förutsäger trädet? → ___

---
## ✂️ Del 2 – Train/Test Split

Innan vi tränar måste vi dela upp datasetet i **träningsdata** och **testdata**.

In [ ]:
# ============================================================
# INTERAKTIV TRAIN/TEST SPLIT
# ============================================================

output_split = widgets.Output()

def visa_split(test_storlek):
    with output_split:
        clear_output(wait=True)
        
        train_pct = 100 - test_storlek
        n_total = len(X)
        n_test = int(n_total * test_storlek / 100)
        n_train = n_total - n_test
        
        # Rita stapeldiagram
        fig, ax = plt.subplots(figsize=(9, 2.5))
        
        ax.barh(['Data'], [n_train], color='#4CAF50', label=f'Träning ({n_train} blommor)', height=0.5)
        ax.barh(['Data'], [n_test], left=n_train, color='#FF9800', 
                 label=f'Test ({n_test} blommor)', height=0.5)
        
        ax.set_xlabel('Antal blommor', fontsize=11)
        ax.set_title(f'Train/Test Split: {train_pct}% träning / {test_storlek}% test', 
                     fontsize=13, fontweight='bold')
        ax.legend(loc='upper right', fontsize=11)
        ax.set_xlim(0, n_total + 10)
        
        # Visa siffror inuti staplarna
        ax.text(n_train/2, 0, f'{n_train} st\n({train_pct}%)', 
                ha='center', va='center', fontweight='bold', fontsize=12, color='white')
        ax.text(n_train + n_test/2, 0, f'{n_test} st\n({test_storlek}%)', 
                ha='center', va='center', fontweight='bold', fontsize=12, color='white')
        
        plt.tight_layout()
        plt.show()
        
        print(f"📊 Med {test_storlek}% testdata:")
        print(f"   Träning: {n_train} blommor  (modellen lär sig från dessa)")
        print(f"   Test:    {n_test} blommor  (vi mäter prestanda på dessa)")
        print()
        
        if test_storlek < 15:
            print("⚠️ Litet testset – osäkra mätresultat!")
        elif test_storlek > 40:
            print("⚠️ Litet träningsset – modellen lär sig lite!")
        else:
            print("✅ Bra split! Tillräckligt med data för både träning och test.")

split_slider = widgets.IntSlider(
    value=20, min=10, max=50, step=5,
    description='Test %:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

widgets.interactive(visa_split, test_storlek=split_slider)

---
### ✏️ Övning 2.1

Prova slidern ovan med olika värden. Svara:

1. Med 20% testdata: Hur många blommor i träningsset? → ___
2. Med 30% testdata: Hur många blommor i testset? → ___
3. Varför är 50% testdata vanligtvis för mycket? → ___

In [ ]:
# ── Gör det riktiga Train/Test Split ─────────────────────
# Vi använder 80% träning och 20% test (standard)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    # 20% test
    random_state=42   # Fixerat slumptal för reproducerbarhet
)

print("✅ Train/Test Split genomfört!")
print(f"   Träningsdata: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"   Testdata:     X_test={X_test.shape},  y_test={y_test.shape}")
print()

# Visa fördelningen av klasser i train och test
print("📊 Klassfördelning:")
print("   Träning:", {cn: (y_train==i).sum() for i,cn in enumerate(class_names)})
print("   Test:   ", {cn: (y_test==i).sum() for i,cn in enumerate(class_names)})

---
## 🌳 Del 3 – Träna modellen!

Nu tränar vi ett **beslutsträd** på träningsdata.  
Klicka på knappen för att träna!

In [ ]:
# ============================================================
# INTERAKTIV TRÄNING MED KNAPP
# ============================================================

output_trana = widgets.Output()
model_container = {}  # Spara modell för senare use

def trana_modell(b):
    with output_trana:
        clear_output(wait=True)
        
        max_d = depth_slider.value
        
        print(f"🔄 Tränar beslutsträd med max_depth={max_d}...")
        print()
        
        # Skapa och träna modellen
        model = DecisionTreeClassifier(max_depth=max_d, random_state=42)
        model.fit(X_train, y_train)
        model_container['model'] = model
        
        # Beräkna accuracy på träning och test
        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc = accuracy_score(y_test, model.predict(X_test))
        
        print(f"✅ Träning klar!")
        print(f"   max_depth = {max_d}")
        print()
        print(f"📊 Resultat:")
        print(f"   🟢 Träningsnoggrannhet: {train_acc:.1%}")
        print(f"   🟠 Testnoggrannhet:     {test_acc:.1%}")
        print()
        
        if train_acc > test_acc + 0.1:
            print("⚠️ Stor skillnad mellan träning och test!")
            print("   Modellen kan ha ÖVERFITTAT (memorerat träningsdata)")
        else:
            print("✅ Träning ≈ Test – modellen generaliserar bra!")
        
        # Rita beslutsträdet
        print()
        print("🌳 Beslutsträdet:")
        fig, ax = plt.subplots(figsize=(12, max(4, max_d * 2)))
        plot_tree(model, feature_names=feature_names, class_names=class_names,
                  filled=True, rounded=True, fontsize=9, ax=ax,
                  impurity=False)
        ax.set_title(f'Beslutsträd (max_depth={max_d})', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

depth_slider = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='max_depth:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

train_button = widgets.Button(
    description='🚀 Träna modellen!',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

train_button.on_click(trana_modell)

display(widgets.VBox([
    widgets.HTML('<b>⚙️ Inställningar:</b>'),
    depth_slider,
    train_button,
    widgets.HTML('<i>💡 max_depth = hur djupt trädet kan bli (fler nivåer = mer komplex modell)</i>'),
    output_trana
]))

---
## 🔬 Del 4 – Experimentera med max_depth

`max_depth` är en **hyperparameter** – en inställning du väljer INNAN träning.

- `max_depth=1` → Enkelt träd, få beslut
- `max_depth=10` → Komplext träd, många beslut

Vad händer med accuracy när du ökar max_depth?

In [ ]:
# ============================================================
# ACCURACY vs MAX_DEPTH
# ============================================================

depth_values = range(1, 11)
train_accuracies = []
test_accuracies = []

for d in depth_values:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_accuracies.append(accuracy_score(y_train, m.predict(X_train)))
    test_accuracies.append(accuracy_score(y_test, m.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(depth_values, [a * 100 for a in train_accuracies], 'o-', 
        color='#4CAF50', linewidth=2, markersize=7, label='Träningsdata')
ax.plot(depth_values, [a * 100 for a in test_accuracies], 's-', 
        color='#FF9800', linewidth=2, markersize=7, label='Testdata')

ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs max_depth', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(list(depth_values))
ax.set_ylim(85, 102)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("📊 Accuracy-tabell:")
print(f"{'max_depth':>10} | {'Träning':>10} | {'Test':>10}")
print("-" * 35)
for d, tr, te in zip(depth_values, train_accuracies, test_accuracies):
    marker = " ← bäst test" if te == max(test_accuracies) and d == test_accuracies.index(max(test_accuracies))+1 else ""
    print(f"{d:>10} | {tr:>9.1%} | {te:>9.1%}{marker}")

---
### ✏️ Övning 4.1 – Analysera grafen

Titta på grafen "Accuracy vs max_depth" ovan och svara:

1. Vid vilket max_depth är testnoggrannheten som HÖGST? → ___
2. Vad händer med träningsnoggrannheten när max_depth ökar? → ___
3. Varför kan hög max_depth vara dåligt? (Tips: tänk på "memorera vs lära sig") → ___
4. Vad är det maximala accuracy man kan nå på testdata? → ___
5. Fyll i tabellen (utan att titta på tabellen ovan – uppskatta från grafen!):
   - max_depth=2, test accuracy ≈ ___ %
   - max_depth=5, test accuracy ≈ ___ %

---
## 🔮 Del 5 – Gör en Förutsägelse!

Nu kan du testa modellen med dina **egna värden**!  
Ange mätningar för en blomma och se vad modellen förutsäger.

In [ ]:
# ============================================================
# INTERAKTIV FÖRUTSÄGELSE
# ============================================================

# Träna en bra modell först (om ingen modell finns)
if 'model' not in model_container:
    model_container['model'] = DecisionTreeClassifier(max_depth=3, random_state=42)
    model_container['model'].fit(X_train, y_train)
    print("✅ Modell tränad (max_depth=3)")

output_pred = widgets.Output()
art_emojis = ['🌸', '🌺', '🌻']

def gora_forutsagelse(sepal_l, sepal_w, petal_l, petal_w):
    with output_pred:
        clear_output(wait=True)
        
        model = model_container['model']
        features = np.array([[sepal_l, sepal_w, petal_l, petal_w]])
        
        pred = model.predict(features)[0]
        proba = model.predict_proba(features)[0]
        
        print(f"🌸 Din blomma:")
        print(f"   Sepal längd: {sepal_l:.1f} cm")
        print(f"   Sepal bredd: {sepal_w:.1f} cm")
        print(f"   Petal längd: {petal_l:.1f} cm")
        print(f"   Petal bredd: {petal_w:.1f} cm")
        print()
        print(f"🤖 Modellens förutsägelse:")
        print(f"   {art_emojis[pred]} **{class_names[pred]}**")
        print()
        print(f"📊 Sannolikheter:")
        for i, (cn, emoji, p) in enumerate(zip(class_names, art_emojis, proba)):
            bar = '█' * int(p * 20)
            marker = " ← VALD" if i == pred else ""
            print(f"   {emoji} {cn:12s}: {p:.1%} {bar}{marker}")

sepal_l_sl = widgets.FloatSlider(value=5.0, min=4.0, max=8.0, step=0.1, 
    description='Sepal längd:', style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='450px'))
sepal_w_sl = widgets.FloatSlider(value=3.0, min=2.0, max=4.5, step=0.1, 
    description='Sepal bredd:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'))
petal_l_sl = widgets.FloatSlider(value=4.0, min=1.0, max=7.0, step=0.1, 
    description='Petal längd:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'))
petal_w_sl = widgets.FloatSlider(value=1.2, min=0.1, max=2.5, step=0.1, 
    description='Petal bredd:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'))

widgets.interactive(gora_forutsagelse, 
                    sepal_l=sepal_l_sl, sepal_w=sepal_w_sl,
                    petal_l=petal_l_sl, petal_w=petal_w_sl)

---
### ✏️ Övning 5.1 – Testa kända blommor

Använd sliders ovan och testa dessa kända blommor:

| Blomma | sepal_l | sepal_w | petal_l | petal_w | Förväntad art | Modellens svar |
|--------|---------|---------|---------|---------|--------------|----------------|
| A | 5.1 | 3.5 | 1.4 | 0.2 | Setosa | ___ |
| B | 6.4 | 3.2 | 4.5 | 1.5 | Versicolor | ___ |
| C | 6.3 | 3.3 | 6.0 | 2.5 | Virginica | ___ |

1. Fick modellen rätt på alla tre? → ___
2. Prova att ändra petal_längd för blomma A till 5.0 – vad händer? → ___

---
## 🧩 Quiz – Lektion 3

**Fråga 1:** Vad är en "hyperparameter" och vad skiljer den från en vanlig parameter?

*Ditt svar:* ___

---

**Fråga 2:** Om en modell har 100% accuracy på träningsdata men 70% på testdata, vad kallas detta problem?

*Ditt svar:* ___

---

**Fråga 3:** Varför sätter vi `random_state=42` i `train_test_split()`?

*Ditt svar:* ___

---

**Fråga 4:** Vad händer om `max_depth=1`? Kan modellen lära sig komplexa mönster?

*Ditt svar:* ___

---

**Fråga 5:** Förklara med egna ord hur ett beslutsträd fungerar.

*Ditt svar:* ___

---

**Fråga 6:** Du har 500 datapunkter. Du väljer test_size=0.2. Hur många datapunkter är i testset?

*Ditt svar:* ___

---

**Fråga 7:** Om du ändrar petal_längd i förutsägelse-widgeten från 1.5 till 5.0 – vad händer med förutsägelsen? Varför?

*Ditt svar:* ___

---

**Fråga 8:** Varför kan vi inte använda testdata för att träna modellen?

*Ditt svar:* ___

---

**Fråga 9:** Vad är accuracy och hur beräknas det?

*Ditt svar:* ___

---

**Fråga 10:** Vilken max_depth verkar optimal för Iris-datasetet? Varför?

*Ditt svar:* ___

---

### ✅ Sammanfattning – Vad du lärt dig:

- 🌳 **Beslutsträd** = En modell som ställer JA/NEJ-frågor om features
- ✂️ **Train/Test Split** = Dela data för att mäta verklig prestanda
- ⚙️ **max_depth** = Hyperparameter som kontrollerar trädbottens komplexitet
- 📊 **Accuracy** = Andel korrekta förutsägelser
- ⚠️ **Overfitting** = Modellen memorerar träningsdata, generaliserare dåligt

**Nästa lektion:** Confusion Matrix – Se VAR modellen gör fel! 🔍